# 第 06 课 · 第一个可运行的二维圆柱绕流程序

> **学习模式：**独立编写 JAX CFD 求解器 | VS Code / Jupyter | 从公式到可运行代码

**本课目标**：组合 JAX/MAC/Navier–Stokes/圆柱惩罚步骤，生成圆柱尾流的速度与涡量图，并检测数值稳定性。

**前置知识**：第 03 至 05 课。

**操作约定**：先逐单元执行，再尝试“实验与练习”；若改变了函数定义或网格设置，请重启内核并执行全部单元格。JAX 第一次调用包含编译时间。

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax import lax
jax.config.update("jax_enable_x64", True)
print("JAX:", jax.__version__, "devices:", jax.devices())

## 1. 模拟配置和严谨的适用范围

域为 $[0,L_x)\times[0,L_y)$ 的**双周期矩形**，中间放置静止圆柱，施加一个均值速度反馈体力保持整体流动。定义：

$$Re=\frac{U_\mathrm{ref}D}{\nu},\qquad D=2R.$$

例如 $D=1, U_\mathrm{ref}=1, \nu=0.01$，对应 $Re=100$。

这是 **周期域圆柱障碍物原型（流向会重复遇到其周期镜像）**，可以学习尾流结构和数值实现，但**不是标准孤立圆柱绕流**：不能直接拿其 $C_D$ 或 Strouhal 数与非周期入口出口文献一一对比。

In [ ]:
def grid(nx=64,ny=32,Lx=8.,Ly=4.):
 dx,dy=Lx/nx,Ly/ny
 x=jnp.arange(nx)*dx; y=jnp.arange(ny)*dy
 X,Y=jnp.meshgrid(x,y,indexing='ij')
 return dx,dy,X,Y

In [ ]:
def div(u,v,dx,dy):
 return (u-jnp.roll(u,1,0))/dx+(v-jnp.roll(v,1,1))/dy

In [ ]:
def grad(p,dx,dy):
 return (jnp.roll(p,-1,0)-p)/dx,(jnp.roll(p,-1,1)-p)/dy

In [ ]:
def lap(q,dx,dy):
 return (jnp.roll(q,-1,0)+jnp.roll(q,1,0)-2*q)/(dx*dx)+(jnp.roll(q,-1,1)+jnp.roll(q,1,1)-2*q)/(dy*dy)

In [ ]:
def symbol(nx,ny,dx,dy):
 kx=2*jnp.pi*jnp.fft.fftfreq(nx,d=dx)[:,None]
 ky=2*jnp.pi*jnp.fft.fftfreq(ny,d=dy)[None,:]
 s=-4*jnp.sin(kx*dx/2)**2/dx**2-4*jnp.sin(ky*dy/2)**2/dy**2
 return jnp.where(s==0,1.,s)

In [ ]:
def solve_poisson(rhs,s):
 rhs_hat=jnp.fft.fftn(rhs)
 p=jnp.fft.ifftn(rhs_hat/s).real
 return p-p.mean()

In [ ]:
def project(u,v,dx,dy,s):
 p=solve_poisson(div(u,v,dx,dy),s)
 px,py=grad(p,dx,dy)
 return u-px,v-py,p

In [ ]:
def advection(u,v,dx,dy):
 v_u=0.25*(v+jnp.roll(v,-1,0)+jnp.roll(v,1,1)+jnp.roll(jnp.roll(v,-1,0),1,1))
 u_v=0.25*(u+jnp.roll(u,1,0)+jnp.roll(u,-1,1)+jnp.roll(jnp.roll(u,1,0),-1,1))
 dux=(jnp.roll(u,-1,0)-jnp.roll(u,1,0))/(2*dx)
 duy=(jnp.roll(u,-1,1)-jnp.roll(u,1,1))/(2*dy)
 dvx=(jnp.roll(v,-1,0)-jnp.roll(v,1,0))/(2*dx)
 dvy=(jnp.roll(v,-1,1)-jnp.roll(v,1,1))/(2*dy)
 return u*dux+v_u*duy,u_v*dvx+v*dvy

In [ ]:
def geometry(nx,ny,dx,dy,cx=2,cy=2,r=.5,eps=None):
 eps=0.8*min(dx,dy) if eps is None else eps
 xu=(jnp.arange(nx)[:,None]+.5)*dx; yu=jnp.arange(ny)[None,:]*dy
 xv=jnp.arange(nx)[:,None]*dx; yv=(jnp.arange(ny)[None,:]+.5)*dy
 # simple nonperiodic distance OK when body away from edges
 ru=jnp.sqrt((xu-cx)**2+(yu-cy)**2)
 rv=jnp.sqrt((xv-cx)**2+(yv-cy)**2)
 masku=.5*(1-jnp.tanh((ru-r)/eps))
 maskv=.5*(1-jnp.tanh((rv-r)/eps))
 return masku,maskv

In [ ]:
def step_navier(u,v,dx,dy,dt,nu,s,masku=None,maskv=None,eta=.02,targetU=0.,body_v=0.,gain=0.):
 def rhs(u,v):
  advu,advv=advection(u,v,dx,dy)
  force=gain*(targetU-jnp.mean(u))
  return -advu+nu*lap(u,dx,dy)+force,-advv+nu*lap(v,dx,dy)
 def filter_and_project(u,v):
  if masku is not None:
   u=u/(1.+dt/eta*masku)
   v=(v+dt/eta*maskv*body_v)/(1.+dt/eta*maskv)
  un,vn,p=project(u,v,dx,dy,s)
  return un,vn,p
 du,dv=rhs(u,v)
 u1,v1,p1=filter_and_project(u+dt*du,v+dt*dv)
 du1,dv1=rhs(u1,v1)
 u2,v2,p=filter_and_project(0.5*(u+u1+dt*du1),0.5*(v+v1+dt*dv1))
 return u2,v2,u1,v1,p

## 2. 参数初始化和对称性破缺

严格对称的初始条件可能在有限时间内停留在对称状态；加入极小的横向扰动，帮助观察非对称尾流的演化。我们只在可视化前进行 $u,v$ 到单元中心的插值。

In [ ]:
nx,ny,Lx,Ly=96,48,8.,4.
dx,dy,X,Y=grid(nx,ny,Lx,Ly)
S=symbol(nx,ny,dx,dy)
cx,cy,R=2.,2.,0.5
mask_u,mask_v=geometry(nx,ny,dx,dy,cx,cy,R)
Re=100.; Uref=1.; nu=Uref*(2*R)/Re
eta=0.025;dt=0.014
u0=jnp.ones((nx,ny))*Uref
xv=jnp.arange(nx)[:,None]*dx; yv=(jnp.arange(ny)[None,:]+.5)*dy
v0=0.002*jnp.exp(-((xv-cx-0.75)**2+(yv-cy-0.3)**2)/0.25)
u0,v0,_=project(u0,v0,dx,dy,S)
print('Re',Re,'dx',dx,'dt',dt,'initial max div',float(jnp.max(jnp.abs(div(u0,v0,dx,dy)))))

In [ ]:
@jax.jit
def advance_chunk(u,v):
 def advance(_,state):
  a,b=state
  un,vn,*_=step_navier(a,b,dx,dy,dt,nu,S,mask_u,mask_v,eta=eta,targetU=Uref,gain=0.5)
  return un,vn
 return lax.fori_loop(0,60,advance,(u,v))

u,v=u0,v0
snapshots=[]
for k in range(12):
 u,v=advance_chunk(u,v)
 snapshots.append((u,v))
 print(f't={(k+1)*60*dt:5.2f} | mean(u)={float(u.mean()):.3f} | max|div|={float(jnp.max(jnp.abs(div(u,v,dx,dy)))):.1e}')
assert bool(jnp.all(jnp.isfinite(u)))
assert float(jnp.max(jnp.abs(div(u,v,dx,dy))))<1e-9

## 3. 绘制速度和涡量

涡量定义为：

$$\omega_z=\frac{\partial v}{\partial x}-\frac{\partial u}{\partial y}.$$

本课程输出一个**教学原型的尾流场**，不能仅凭图像判断涡脱落频率、网格无关性或受力精度。

In [ ]:
def face_to_cell(u,v):
 return 0.5*(u+jnp.roll(u,1,0)),0.5*(v+jnp.roll(v,1,1))

def vorticity(u,v,dx,dy):
 uc,vc=face_to_cell(u,v)
 return (jnp.roll(vc,-1,0)-jnp.roll(vc,1,0))/(2*dx)-(jnp.roll(uc,-1,1)-jnp.roll(uc,1,1))/(2*dy)

uc,vc=face_to_cell(u,v)
w=vorticity(u,v,dx,dy)
fig,axs=plt.subplots(2,1,figsize=(12,6),constrained_layout=True)
for ax,field,title,cmap in zip(axs,[uc,w],['streamwise velocity u','vorticity omega_z'],['viridis','RdBu_r']):
 im=ax.imshow(np.asarray(field).T,origin='lower',extent=(0,Lx,0,Ly),aspect='equal',cmap=cmap)
 circle=plt.Circle((cx,cy),R,color='white',ec='black',lw=1.2);ax.add_patch(circle)
 ax.set_title(title);ax.set_xlabel('x');ax.set_ylabel('y');fig.colorbar(im,ax=ax,shrink=.8)
plt.show()

## 4. 实验与作业

1. 将网格从 $96\times48$ 改成 $128\times64$，并相应减小时间步长。比较尾流结构。
2. 把计算延长到 $t\ge 30$，观察是否出现规律性非对称振荡；**观察到振荡也不等于已验证经典圆柱涡街**。
3. 改变 $Re=40,100,150$，描述图像的定性差异，并检查 CFL、能量、散度、是否出现 `NaN`。
4. **重点思考**：为什么周期流向条件导致下游尾流再次进入上游？标准入口/出口应当怎样设置？

**本课通过条件**：模拟完成、速度和涡量图可见、零散度残差极小、你能明确说明周期边界的物理意义。